# Javis

Esta é uma **alternativa em notebook** para quem gosta de trabalhar no Jupyter. O modo normal de uso continua sendo o aplicativo `Javis.pyw` (a interface em vidro).

> **Aviso:** o notebook **não** grava nada no arquivo de dados. Ele só lê o estado e mostra um painel, para você consultar o plano sem abrir a janela. Gravar continua sendo feito pela interface, que pede confirmação antes de qualquer mudança.

Se o arquivo `estado.json` não existir ainda, abra o `Javis.pyw` uma vez para que ele seja criado.

In [ ]:
import json
import sys
from pathlib import Path

# O pacote do Javis mora ao lado deste arquivo.
ROOT = Path.cwd()
if not (ROOT / "javis").is_dir():
    ROOT = Path(r"C:\Users\leosi\OneDrive\Documentos\Javis")
sys.path.insert(0, str(ROOT))

from javis.config import DATA_DIR, PLAN_FILE, STATE_FILE
from javis.storage import br_date, br_datetime, load_state

print("Dados em:", DATA_DIR)
print("Estado:", STATE_FILE, "existe" if STATE_FILE.exists() else "ainda não criado")

## Painel

Um resumo do plano, das decisões vigentes e das ações abertas.

In [ ]:
estado, aviso = load_state()

if aviso:
    print("Aviso:", aviso)

print("=" * 60)
print("PLANO:", estado.get("projeto"))
print("Objetivo:", estado.get("objetivo") or "(ainda não definido)")
print("=" * 60)

print("\nPLANO ATUAL")
print("-" * 60)
print(estado.get("plano", "(vazio)"))

print("\nDECISÕES VIGENTES")
print("-" * 60)
vigentes = [d for d in estado.get("decisoes", []) if d.get("status", "vigente") == "vigente"]
if vigentes:
    for d in vigentes:
        print(f"- {d.get('id')} ({br_date(d.get('data'))}): {d.get('texto')}")
        if d.get("motivo"):
            print(f"    motivo: {d['motivo']}")
else:
    print("(nenhuma decisão registrada)")

print("\nAÇÕES ABERTAS")
print("-" * 60)
abertas = [a for a in estado.get("acoes", []) if a.get("status") in {"a fazer", "em andamento"}]
if abertas:
    for a in abertas:
        print(f"- {a.get('id')} [{a.get('prioridade', 'média')}] {a.get('texto')}")
        print(f"    esperado: {a.get('resultado_esperado') or '—'}")
        if a.get("revisar_em") or a.get("prazo"):
            print(f"    rever em: {br_date(a.get('revisar_em') or a.get('prazo'))}")
        for r in a.get("resultados", []):
            print(f"    • {br_date(r.get('data'))}: {r.get('texto')}")
else:
    print("(nenhuma ação aberta)")

## Registros recentes

Os últimos registros salvos pela interface.

In [ ]:
print("REGISTROS RECENTES")
print("-" * 60)
notas = estado.get("notas", [])[-12:]
if notas:
    for n in reversed(notas):
        texto = " ".join(str(n.get("texto", "")).split())
        print(f"- {n.get('id')} · {br_datetime(n.get('data'))}")
        print(f"    {texto[:220]}")
else:
    print("(nenhum registro salvo)")

## Perguntar ao Javis local (somente leitura)

Esta célula envia a pergunta ao Ollama local com o contexto do plano. **Não grava nada** — é só para experimentar a análise fora da interface.

In [ ]:
import urllib.request

from javis.config import CTX_QUICK, OLLAMA_URL, PREDICT_QUICK
from javis.llm import SYSTEM_QUICK, build_context, compact_json

def perguntar(pergunta: str, modelo: str = "qwen3:1.7b") -> str:
    """Envia a pergunta ao Ollama local e devolve a resposta (sem salvar)."""
    budget = __import__("javis.llm", fromlist=["ContextBudget"]).ContextBudget(CTX_QUICK, PREDICT_QUICK, SYSTEM_QUICK)
    ctx = build_context(estado, estado.get("plano", ""), None, False, budget)
    corpo = {
        "model": modelo, "stream": False, "think": False,
        "options": {"temperature": 0.2, "num_predict": PREDICT_QUICK, "num_ctx": CTX_QUICK},
        "messages": [
            {"role": "system", "content": SYSTEM_QUICK},
            {"role": "user", "content": "CONTEXTO LOCAL:\n" + compact_json(ctx) + "\n\nPEDIDO:\n" + pergunta},
        ],
    }
    req = urllib.request.Request(OLLAMA_URL, data=json.dumps(corpo, ensure_ascii=False).encode("utf-8"),
                                headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=240) as r:
        return json.loads(r.read().decode("utf-8"))["message"]["content"]

print(perguntar("Qual é o próximo passo mais importante do plano?"))